In [1]:
import numpy as np
import pandas as pd
import warnings

from itertools import combinations
from math import comb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

### File Load

In [2]:
df = pd.read_csv(r"../../data/processed/data_vif.csv")
target = "Risk_Label"

n_total = len(df)
n_train = int(n_total * 0.5)
n_valid = int(n_total * 0.3)
n_test = n_total - n_train - n_valid

#Date 컬럼 제거
df.drop("Date", axis=1, inplace=True)

# Risk_Label 매핑 (High Risk=1, Low Risk=0)
df["Risk_Label"] = df["Risk_Label"].map({"Low Risk": 0, "High Risk": 1})

#train/valid/test 분할 5/3/2
train_df = df.iloc[:n_train].reset_index(drop=True)
valid_df = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
test_df = df.iloc[n_train + n_valid :].reset_index(drop=True)

print(f"total rows: {n_total}, train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

total rows: 4108, train: 2054, valid: 1232, test: 822
train shape: (2054, 28) (2054,)
valid shape: (1232, 28) (1232,)
test shape: (822, 28) (822,)


## Min-Max Scaling

In [3]:
scaler = MinMaxScaler().set_output(transform="pandas")

# train에만 fit하고 valid/test에는 같은 scaler를 적용
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print('train/valid/test:', len(X_train), len(X_valid), len(X_test))
print('y_train class:', y_train.value_counts().to_dict())
print('y_valid class:', y_valid.value_counts().to_dict())
print('y_test class:', y_test.value_counts().to_dict())

train/valid/test: 2054 1232 822
y_train class: {0: 1859, 1: 195}
y_valid class: {0: 1070, 1: 162}
y_test class: {0: 730, 1: 92}


## 파라미터, 지표

In [15]:
# ============================================================
# 1. Metric 함수
# ============================================================

def gmean_score(y_true, y_pred):
    """
    binary label 기준:
    0 = Low Risk
    1 = High Risk

    G-Mean = sqrt(Recall * Specificity)
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return np.sqrt(recall * specificity)


def classification_metrics(y_true, y_pred, y_prob=None):
    """
    각종 성능지표 계산.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall,
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "gmean": np.sqrt(recall * specificity),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    if y_prob is not None:
        try:
            metrics["auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            metrics["auc"] = np.nan
    else:
        metrics["auc"] = np.nan

    return metrics


def find_best_threshold(y_true, y_prob, threshold_grid=None):
    """
    valid set에서 G-Mean이 최대가 되는 threshold 탐색.
    """
    if threshold_grid is None:
        threshold_grid = np.arange(0.05, 0.96, 0.05)

    best_threshold = 0.5
    best_metrics = None
    best_pred = None

    for threshold in threshold_grid:
        y_pred = (y_prob >= threshold).astype(int)
        metrics = classification_metrics(y_true, y_pred, y_prob)

        if best_metrics is None:
            best_threshold = threshold
            best_metrics = metrics
            best_pred = y_pred
        else:
            old_key = (
                best_metrics["gmean"],
                best_metrics["recall"],
                best_metrics["accuracy"],
                best_metrics["specificity"],
                best_metrics["f1"]
            )
            new_key = (
                metrics["gmean"],
                metrics["recall"],
                metrics["accuracy"],
                metrics["specificity"],
                metrics["f1"]
            )

            if new_key > old_key:
                best_threshold = threshold
                best_metrics = metrics
                best_pred = y_pred

    return best_threshold, best_metrics, best_pred


# ============================================================
# 2. LogisticRegression 하이퍼파라미터 후보
# ============================================================

def make_logistic_l2_param_grid(
    C_grid=None,
    solver_grid=None,
    class_weight_grid=None,
    random_state=1
):
    """
    전진/후진/Stepwise/전역 선택법용 LogisticRegression 후보.
    변수선택은 wrapper가 담당하므로 penalty는 L2로 고정.
    """

    if C_grid is None:
        C_grid = [0.001, 0.01, 0.1, 1, 10, 100]

    if solver_grid is None:
        solver_grid = ["liblinear"]

    if class_weight_grid is None:
        # ADASYN 전이면 불균형 데이터이므로 balanced도 후보에 넣는 게 맞음
        class_weight_grid = ["balanced"]

    param_grid = []

    for C in C_grid:
        for solver in solver_grid:
            for class_weight in class_weight_grid:
                param_grid.append({
                    "penalty": "l2",
                    "C": C,
                    "solver": solver,
                    "class_weight": class_weight,
                    "max_iter": 10000,
                    "random_state": random_state
                })

    return param_grid


def build_logistic_pipeline(params, scaler_type="standard"):
    """
    scaler + LogisticRegression pipeline 생성.
    scaler_type:
    - "standard"
    - "minmax"
    - None
    """

    steps = []

    if scaler_type == "standard":
        steps.append(("scaler", StandardScaler()))
    elif scaler_type == "minmax":
        steps.append(("scaler", MinMaxScaler()))
    elif scaler_type is None:
        pass
    else:
        raise ValueError("scaler_type은 'standard', 'minmax', None 중 하나여야 함.")

    steps.append(("model", LogisticRegression(**params)))

    return Pipeline(steps)


# ============================================================
# 3. AIC/BIC, coef 계산
# ============================================================

def calculate_aic_bic(fitted_model, X, y, n_features):
    """
    LogisticRegression 기준 approximate AIC/BIC.
    규제 로지스틱에서 완전한 classical AIC/BIC는 아니고 비교용 지표로 사용.
    """
    proba = fitted_model.predict_proba(X)
    proba = np.clip(proba, 1e-15, 1 - 1e-15)

    y_arr = np.asarray(y).astype(int)
    ll = np.sum(
        y_arr * np.log(proba[:, 1]) +
        (1 - y_arr) * np.log(proba[:, 0])
    )

    n = len(y_arr)
    k = n_features + 1

    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll

    return aic, bic


def get_coef_df(fitted_model, features):
    """
    pipeline 안의 LogisticRegression 계수 추출.
    """
    logit = fitted_model.named_steps["model"]
    coef = logit.coef_.ravel()

    coef_df = pd.DataFrame({
        "feature": features,
        "coef": coef,
        "abs_coef": np.abs(coef)
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

    return coef_df


# ============================================================
# 4. 특정 변수 조합에 대해 Logistic 하이퍼파라미터 서칭
# ============================================================

def evaluate_feature_subset(
    X_train,
    y_train,
    X_valid,
    y_valid,
    features,
    param_grid=None,
    scaler_type="standard",
    threshold_search=True,
    threshold_grid=None,
    random_state=1
):
    """
    특정 feature subset에 대해:
    train 학습 → valid 검증 → LogisticRegression hyperparameter search.
    선택 기준은 valid G-Mean.
    """

    if param_grid is None:
        param_grid = make_logistic_l2_param_grid(random_state=random_state)

    X_tr = X_train[features]
    X_va = X_valid[features]

    records = []

    best_model = None
    best_params = None
    best_threshold = 0.5
    best_metrics = None

    for params in param_grid:
        try:
            model = build_logistic_pipeline(params, scaler_type=scaler_type)
            model.fit(X_tr, y_train)

            y_prob = model.predict_proba(X_va)[:, 1]

            if threshold_search:
                threshold, metrics, y_pred = find_best_threshold(
                    y_valid,
                    y_prob,
                    threshold_grid=threshold_grid
                )
            else:
                threshold = 0.5
                y_pred = model.predict(X_va)
                metrics = classification_metrics(y_valid, y_pred, y_prob)

            aic, bic = calculate_aic_bic(
                model,
                X_tr,
                y_train,
                n_features=len(features)
            )

            row = {
                "features": list(features),
                "n_features": len(features),
                "threshold": threshold,
                "C": params["C"],
                "solver": params["solver"],
                "class_weight": params["class_weight"],
                "valid_accuracy": metrics["accuracy"],
                "valid_precision": metrics["precision"],
                "valid_recall": metrics["recall"],
                "valid_specificity": metrics["specificity"],
                "valid_f1": metrics["f1"],
                "valid_gmean": metrics["gmean"],
                "valid_auc": metrics["auc"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "tp": metrics["tp"],
                "aic": aic,
                "bic": bic
            }

            records.append(row)

            if best_metrics is None:
                best_model = model
                best_params = params
                best_threshold = threshold
                best_metrics = metrics
            else:
                old_key = (
                    best_metrics["gmean"],
                    best_metrics["recall"],
                    best_metrics["accuracy"],
                    best_metrics["specificity"],
                    best_metrics["f1"]
                )
                new_key = (
                    metrics["gmean"],
                    metrics["recall"],
                    metrics["accuracy"],
                    metrics["specificity"],
                    metrics["f1"]
                )

                if new_key > old_key:
                    best_model = model
                    best_params = params
                    best_threshold = threshold
                    best_metrics = metrics

        except Exception as e:
            records.append({
                "features": list(features),
                "n_features": len(features),
                "C": params.get("C", None),
                "solver": params.get("solver", None),
                "class_weight": params.get("class_weight", None),
                "error": str(e)
            })

    result_df = pd.DataFrame(records)

    if "valid_gmean" in result_df.columns:
        result_df = result_df.dropna(subset=["valid_gmean"])
        result_df = result_df.sort_values(
            by=["valid_gmean", "valid_recall", "valid_accuracy", "aic", "bic"],
            ascending=[False, False, False, True, True]
        ).reset_index(drop=True)

    if best_model is None:
        raise RuntimeError("모든 LogisticRegression 후보가 실패함.")

    return {
        "model": best_model,
        "params": best_params,
        "threshold": best_threshold,
        "metrics": best_metrics,
        "result_df": result_df
    }


# ============================================================
# 5. 결과 출력 함수
# ============================================================

def print_selection_summary(result, title="Selection Result"):
    print("=" * 80)
    print(title)
    print("=" * 80)

    print(f"선택 변수 개수: {len(result['selected_features'])}")
    print("선택 변수:")
    print(result["selected_features"])

    print("-" * 80)
    print("Best LogisticRegression Params")
    print(result["best_params"])

    print("-" * 80)
    print(f"Best Threshold: {result['best_threshold']:.2f}")

    m = result["valid_metrics"]

    print("-" * 80)
    print("Validation Metrics")
    print(f"Accuracy   : {m['accuracy']:.4f}")
    print(f"Precision  : {m['precision']:.4f}")
    print(f"Recall     : {m['recall']:.4f}")
    print(f"Specificity: {m['specificity']:.4f}")
    print(f"F1         : {m['f1']:.4f}")
    print(f"G-Mean     : {m['gmean']:.4f}")
    print(f"AUC        : {m['auc']:.4f}" if not np.isnan(m["auc"]) else "AUC        : nan")

    print("-" * 80)
    print("Confusion Matrix")
    print(f"TN: {m['tn']}, FP: {m['fp']}")
    print(f"FN: {m['fn']}, TP: {m['tp']}")
    print("=" * 80)

## Lasso

L1 Logistic Regression 기반 변수선택

C=1/lambda (규제 강도)

C가 작을수록 강한 규제 -> 적은 변수 선택

C가 클수록 약한 규제 -> 많은 변수 선택

In [5]:
def select_lasso_features(
    X_train,
    y_train,
    X_valid,
    y_valid,
    C_grid=None,
    threshold_grid=None,
    cv_splits=5,
    coef_tol=1e-6,
    class_weight=None,
    max_iter=20000,
    random_state=1
):
    # 기본값 처리
    if C_grid is None:
        C_grid = [0.001, 0.01, 0.1, 1, 10]
    if threshold_grid is None:
        threshold_grid = np.arange(0.1, 0.91, 0.05)

    tscv = TimeSeriesSplit(n_splits=cv_splits)
    results = []

    for C in C_grid:
        model = LogisticRegression(
            penalty="l1",
            solver="saga",
            C=C,
            class_weight=class_weight,
            max_iter=max_iter,
            random_state=random_state,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        valid_proba = model.predict_proba(X_valid)[:, 1]

        for threshold in threshold_grid:
            valid_pred = (valid_proba >= threshold).astype(int)
            gmean = gmean_score(y_valid, valid_pred)
            results.append({
                "C": C,
                "threshold": threshold,
                "gmean": gmean
            })

    result_df = pd.DataFrame(results)
    best_row = result_df.loc[result_df["gmean"].idxmax()]
    best_C = best_row["C"]
    best_threshold = best_row["threshold"]

    # 최종 모델 학습
    final_model = LogisticRegression(
        penalty="l1",
        solver="saga",
        C=best_C,
        class_weight=class_weight,
        max_iter=max_iter,
        random_state=1,
        n_jobs=-1
    )
    final_model.fit(X_train, y_train)
    coef = final_model.coef_.ravel()
    selected_features = X_train.columns[np.abs(coef) > coef_tol].tolist()
    coef_df = pd.DataFrame({
        "feature": X_train.columns,
        "coef": coef,
        "abs_coef": np.abs(coef)
    }).sort_values("abs_coef", ascending=False)

    return {
        "selected_features": selected_features,
        "best_C": best_C,
        "best_threshold": best_threshold,
        "cv_result": result_df,
        "coef_df": coef_df,
        "model": final_model
    }
    
    
def calculate_aic_bic(fitted_model, X, y, n_features):
    """
    LogisticRegression 기준 approximate AIC/BIC.
    규제 로지스틱에서 완전한 classical AIC/BIC는 아니고 비교용 지표로 사용.
    """
    proba = fitted_model.predict_proba(X)
    proba = np.clip(proba, 1e-15, 1 - 1e-15)

    y_arr = np.asarray(y).astype(int)
    ll = np.sum(
        y_arr * np.log(proba[:, 1]) +
        (1 - y_arr) * np.log(proba[:, 0])
    )

    n = len(y_arr)
    k = n_features + 1

    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll

    return aic, bic


def get_coef_df(fitted_model, features):
    """
    pipeline 안의 LogisticRegression 계수 추출.
    """
    logit = fitted_model.named_steps["model"]
    coef = logit.coef_.ravel()

    coef_df = pd.DataFrame({
        "feature": features,
        "coef": coef,
        "abs_coef": np.abs(coef)
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

    return coef_df


C는 1/Lambda , LogisticRegression의 threshold, coef_total을 하이퍼 파라미터로 둠

In [6]:
C_grid = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]

lasso_result = select_lasso_features(
    X_train,
    y_train,
    X_valid,
    y_valid,
    C_grid=C_grid,
    cv_splits=5
)

lasso_features = lasso_result["selected_features"]
best_C = lasso_result["best_C"]
lasso_cv_result = lasso_result["cv_result"]
lasso_coef_df = lasso_result["coef_df"]
lasso_model = lasso_result["model"]

print("Best C:", best_C)
display("Selected features:", lasso_features)

display(lasso_cv_result.sort_values("gmean", ascending=False).head(5))
display(lasso_coef_df)

NameError: name 'TimeSeriesSplit' is not defined

## 전역선택법

G-mean기준, tie breaker: AIC,BIC

In [7]:
def global_exhaustive_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    min_features=2,
    max_features=7,
    scaler_type="standard",
    threshold_search=True,
    threshold_grid=None,
    param_grid=None,
    random_state=1,
    verbose=True
):
    """
    전역선택법 Exhaustive Search + LogisticRegression(L2) 하이퍼파라미터 서칭.
    모든 feature subset을 평가.
    train으로 학습하고 valid로 검증.
    선택 기준은 valid G-Mean.
    """

    if param_grid is None:
        param_grid = make_logistic_l2_param_grid(random_state=random_state)

    feature_names = list(X_train.columns)
    p = len(feature_names)

    max_features = min(max_features, p)
    min_features = max(1, min_features)

    total_combos = sum(
        comb(p, k)
        for k in range(min_features, max_features + 1)
    )

    print("=" * 80)
    print("전역선택법 Global Exhaustive Selection + Logistic Hyperparameter Search")
    print("=" * 80)
    print(f"전체 변수 개수: {p}")
    print(f"평가 변수 개수 범위: {min_features} ~ {max_features}")
    print(f"총 subset 조합 수: {total_combos:,}")
    print(f"Logistic parameter 후보 수: {len(param_grid)}")
    print("=" * 80)

    records = []

    best_features = None
    best_model = None
    best_params = None
    best_threshold = 0.5
    best_metrics = None
    best_aic = None
    best_bic = None

    for k in range(min_features, max_features + 1):
        iterator = combinations(feature_names, k)

        for subset in tqdm(iterator, total=comb(p, k), desc=f"{k} features"):
            subset = list(subset)

            eval_result = evaluate_feature_subset(
                X_train,
                y_train,
                X_valid,
                y_valid,
                features=subset,
                param_grid=param_grid,
                scaler_type=scaler_type,
                threshold_search=threshold_search,
                threshold_grid=threshold_grid,
                random_state=random_state
            )

            top_row = eval_result["result_df"].iloc[0].to_dict()
            metrics = eval_result["metrics"]
            params = eval_result["params"]

            records.append({
                "features": subset,
                "n_features": len(subset),
                "valid_gmean": metrics["gmean"],
                "valid_accuracy": metrics["accuracy"],
                "valid_precision": metrics["precision"],
                "valid_recall": metrics["recall"],
                "valid_specificity": metrics["specificity"],
                "valid_f1": metrics["f1"],
                "valid_auc": metrics["auc"],
                "threshold": eval_result["threshold"],
                "C": params["C"],
                "solver": params["solver"],
                "class_weight": params["class_weight"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "tp": metrics["tp"],
                "aic": top_row["aic"],
                "bic": top_row["bic"]
            })

            if best_metrics is None:
                best_features = subset
                best_model = eval_result["model"]
                best_params = params
                best_threshold = eval_result["threshold"]
                best_metrics = metrics
                best_aic = top_row["aic"]
                best_bic = top_row["bic"]
            else:
                old_key = (
                    best_metrics["gmean"],
                    best_metrics["recall"],
                    best_metrics["accuracy"],
                    -best_aic,
                    -best_bic
                )
                new_key = (
                    metrics["gmean"],
                    metrics["recall"],
                    metrics["accuracy"],
                    -top_row["aic"],
                    -top_row["bic"]
                )

                if new_key > old_key:
                    best_features = subset
                    best_model = eval_result["model"]
                    best_params = params
                    best_threshold = eval_result["threshold"]
                    best_metrics = metrics
                    best_aic = top_row["aic"]
                    best_bic = top_row["bic"]

    result_df = pd.DataFrame(records)

    result_df = result_df.sort_values(
        by=["valid_gmean", "valid_recall", "valid_accuracy", "aic", "bic", "n_features"],
        ascending=[False, False, False, True, True, True]
    ).reset_index(drop=True)

    coef_df = get_coef_df(best_model, best_features)

    return {
        "method": "global_exhaustive",
        "selected_features": best_features,
        "best_n_features": len(best_features),
        "result_df": result_df,
        "model": best_model,
        "best_params": best_params,
        "best_threshold": best_threshold,
        "valid_metrics": best_metrics,
        "best_aic": best_aic,
        "best_bic": best_bic,
        "coef_df": coef_df
    }

## 전진선택

In [8]:
def forward_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    scaler_type="standard",
    threshold_search=True,
    threshold_grid=None,
    param_grid=None,
    min_delta=0.0,
    max_features=None,
    random_state=1,
    verbose=True
):
    """
    전진선택법 + LogisticRegression(L2) 하이퍼파라미터 서칭.
    train으로 학습하고 valid로 검증.
    선택 기준은 valid G-Mean.
    """

    if param_grid is None:
        param_grid = make_logistic_l2_param_grid(random_state=random_state)

    remaining_features = list(X_train.columns)
    selected_features = []

    current_score = 0.0
    scores = []
    history = []
    candidate_records = []

    best_model = None
    best_params = None
    best_threshold = 0.5
    best_metrics = None

    print("=" * 80)
    print("전진선택법 Forward Selection + Logistic Hyperparameter Search")
    print("=" * 80)

    step = 0

    while len(remaining_features) > 0:
        step += 1

        if max_features is not None and len(selected_features) >= max_features:
            print("\nmax_features 도달. 종료.")
            break

        if verbose:
            print(f"\n[Step {step}] 현재 선택 변수 {len(selected_features)}개 / 남은 변수 {len(remaining_features)}개")
            print(f"현재 Best Valid G-Mean: {current_score:.4f}")

        step_best_feature = None
        step_best_eval = None
        step_best_score = current_score

        for feature in remaining_features:
            test_features = selected_features + [feature]

            eval_result = evaluate_feature_subset(
                X_train,
                y_train,
                X_valid,
                y_valid,
                features=test_features,
                param_grid=param_grid,
                scaler_type=scaler_type,
                threshold_search=threshold_search,
                threshold_grid=threshold_grid,
                random_state=random_state
            )

            metrics = eval_result["metrics"]
            params = eval_result["params"]

            candidate_records.append({
                "step": step,
                "action": "try_add",
                "candidate_feature": feature,
                "features": list(test_features),
                "n_features": len(test_features),
                "valid_gmean": metrics["gmean"],
                "valid_accuracy": metrics["accuracy"],
                "valid_precision": metrics["precision"],
                "valid_recall": metrics["recall"],
                "valid_specificity": metrics["specificity"],
                "valid_f1": metrics["f1"],
                "valid_auc": metrics["auc"],
                "threshold": eval_result["threshold"],
                "C": params["C"],
                "solver": params["solver"],
                "class_weight": params["class_weight"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "tp": metrics["tp"]
            })

            if metrics["gmean"] > step_best_score + min_delta:
                step_best_feature = feature
                step_best_eval = eval_result
                step_best_score = metrics["gmean"]

        if step_best_feature is not None:
            selected_features.append(step_best_feature)
            remaining_features.remove(step_best_feature)

            best_model = step_best_eval["model"]
            best_params = step_best_eval["params"]
            best_threshold = step_best_eval["threshold"]
            best_metrics = step_best_eval["metrics"]

            current_score = best_metrics["gmean"]
            scores.append(current_score)

            history.append({
                "step": step,
                "action": "add",
                "feature": step_best_feature,
                "features": list(selected_features),
                "n_features": len(selected_features),
                "valid_gmean": best_metrics["gmean"],
                "valid_accuracy": best_metrics["accuracy"],
                "valid_precision": best_metrics["precision"],
                "valid_recall": best_metrics["recall"],
                "valid_specificity": best_metrics["specificity"],
                "valid_f1": best_metrics["f1"],
                "valid_auc": best_metrics["auc"],
                "threshold": best_threshold,
                "C": best_params["C"],
                "solver": best_params["solver"],
                "class_weight": best_params["class_weight"],
                "tn": best_metrics["tn"],
                "fp": best_metrics["fp"],
                "fn": best_metrics["fn"],
                "tp": best_metrics["tp"]
            })

            print(
                f"  ✓ 추가: {step_best_feature} | "
                f"G-Mean: {best_metrics['gmean']:.4f} | "
                f"Recall: {best_metrics['recall']:.4f} | "
                f"Specificity: {best_metrics['specificity']:.4f} | "
                f"C: {best_params['C']} | solver: {best_params['solver']} | "
                f"class_weight: {best_params['class_weight']} | "
                f"threshold: {best_threshold:.2f}"
            )

        else:
            print("  ✗ 추가해도 G-Mean 개선 없음. 종료.")
            break

    history_df = pd.DataFrame(history)
    candidate_df = pd.DataFrame(candidate_records)

    if len(candidate_df) > 0:
        candidate_df = candidate_df.sort_values(
            by=["valid_gmean", "valid_recall", "valid_accuracy"],
            ascending=False
        ).reset_index(drop=True)

    coef_df = get_coef_df(best_model, selected_features) if best_model is not None else pd.DataFrame()

    return {
        "method": "forward",
        "selected_features": selected_features,
        "scores": scores,
        "history_df": history_df,
        "candidate_df": candidate_df,
        "model": best_model,
        "best_params": best_params,
        "best_threshold": best_threshold,
        "valid_metrics": best_metrics,
        "coef_df": coef_df
    }

## 후진선택 

In [9]:
def backward_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    scaler_type="standard",
    threshold_search=True,
    threshold_grid=None,
    param_grid=None,
    min_delta=0.0,
    min_features=1,
    random_state=1,
    verbose=True
):
    """
    후진선택법 + LogisticRegression(L2) 하이퍼파라미터 서칭.
    전체 변수에서 시작해서 하나씩 제거.
    train으로 학습하고 valid로 검증.
    선택 기준은 valid G-Mean.
    """

    if param_grid is None:
        param_grid = make_logistic_l2_param_grid(random_state=random_state)

    selected_features = list(X_train.columns)
    removed_features = []

    history = []
    candidate_records = []

    print("=" * 80)
    print("후진선택법 Backward Selection + Logistic Hyperparameter Search")
    print("=" * 80)

    # Initial full model
    initial_eval = evaluate_feature_subset(
        X_train,
        y_train,
        X_valid,
        y_valid,
        features=selected_features,
        param_grid=param_grid,
        scaler_type=scaler_type,
        threshold_search=threshold_search,
        threshold_grid=threshold_grid,
        random_state=random_state
    )

    best_model = initial_eval["model"]
    best_params = initial_eval["params"]
    best_threshold = initial_eval["threshold"]
    best_metrics = initial_eval["metrics"]

    current_score = best_metrics["gmean"]
    scores = [current_score]

    history.append({
        "step": 0,
        "action": "initial",
        "feature": None,
        "features": list(selected_features),
        "n_features": len(selected_features),
        "valid_gmean": best_metrics["gmean"],
        "valid_accuracy": best_metrics["accuracy"],
        "valid_precision": best_metrics["precision"],
        "valid_recall": best_metrics["recall"],
        "valid_specificity": best_metrics["specificity"],
        "valid_f1": best_metrics["f1"],
        "valid_auc": best_metrics["auc"],
        "threshold": best_threshold,
        "C": best_params["C"],
        "solver": best_params["solver"],
        "class_weight": best_params["class_weight"],
        "tn": best_metrics["tn"],
        "fp": best_metrics["fp"],
        "fn": best_metrics["fn"],
        "tp": best_metrics["tp"]
    })

    print(f"\n[Initial] 전체 변수 {len(selected_features)}개")
    print(
        f"초기 G-Mean: {current_score:.4f} | "
        f"Recall: {best_metrics['recall']:.4f} | "
        f"Specificity: {best_metrics['specificity']:.4f}"
    )

    step = 0

    while len(selected_features) > min_features:
        step += 1

        if verbose:
            print(f"\n[Step {step}] 현재 변수 {len(selected_features)}개")

        step_remove_feature = None
        step_best_eval = None
        step_best_score = current_score

        for feature in selected_features:
            test_features = [f for f in selected_features if f != feature]

            eval_result = evaluate_feature_subset(
                X_train,
                y_train,
                X_valid,
                y_valid,
                features=test_features,
                param_grid=param_grid,
                scaler_type=scaler_type,
                threshold_search=threshold_search,
                threshold_grid=threshold_grid,
                random_state=random_state
            )

            metrics = eval_result["metrics"]
            params = eval_result["params"]

            candidate_records.append({
                "step": step,
                "action": "try_remove",
                "candidate_feature": feature,
                "features": list(test_features),
                "n_features": len(test_features),
                "valid_gmean": metrics["gmean"],
                "valid_accuracy": metrics["accuracy"],
                "valid_precision": metrics["precision"],
                "valid_recall": metrics["recall"],
                "valid_specificity": metrics["specificity"],
                "valid_f1": metrics["f1"],
                "valid_auc": metrics["auc"],
                "threshold": eval_result["threshold"],
                "C": params["C"],
                "solver": params["solver"],
                "class_weight": params["class_weight"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "tp": metrics["tp"]
            })

            if metrics["gmean"] > step_best_score + min_delta:
                step_remove_feature = feature
                step_best_eval = eval_result
                step_best_score = metrics["gmean"]

        if step_remove_feature is not None:
            selected_features.remove(step_remove_feature)
            removed_features.append(step_remove_feature)

            best_model = step_best_eval["model"]
            best_params = step_best_eval["params"]
            best_threshold = step_best_eval["threshold"]
            best_metrics = step_best_eval["metrics"]

            current_score = best_metrics["gmean"]
            scores.append(current_score)

            history.append({
                "step": step,
                "action": "remove",
                "feature": step_remove_feature,
                "features": list(selected_features),
                "n_features": len(selected_features),
                "valid_gmean": best_metrics["gmean"],
                "valid_accuracy": best_metrics["accuracy"],
                "valid_precision": best_metrics["precision"],
                "valid_recall": best_metrics["recall"],
                "valid_specificity": best_metrics["specificity"],
                "valid_f1": best_metrics["f1"],
                "valid_auc": best_metrics["auc"],
                "threshold": best_threshold,
                "C": best_params["C"],
                "solver": best_params["solver"],
                "class_weight": best_params["class_weight"],
                "tn": best_metrics["tn"],
                "fp": best_metrics["fp"],
                "fn": best_metrics["fn"],
                "tp": best_metrics["tp"]
            })

            print(
                f"  ✗ 제거: {step_remove_feature} | "
                f"G-Mean: {best_metrics['gmean']:.4f} | "
                f"Recall: {best_metrics['recall']:.4f} | "
                f"Specificity: {best_metrics['specificity']:.4f} | "
                f"C: {best_params['C']} | solver: {best_params['solver']} | "
                f"class_weight: {best_params['class_weight']} | "
                f"threshold: {best_threshold:.2f}"
            )

        else:
            print("  ✓ 제거해도 G-Mean 개선 없음. 종료.")
            break

    history_df = pd.DataFrame(history)
    candidate_df = pd.DataFrame(candidate_records)

    if len(candidate_df) > 0:
        candidate_df = candidate_df.sort_values(
            by=["valid_gmean", "valid_recall", "valid_accuracy"],
            ascending=False
        ).reset_index(drop=True)

    coef_df = get_coef_df(best_model, selected_features)

    return {
        "method": "backward",
        "selected_features": selected_features,
        "removed_features": removed_features,
        "scores": scores,
        "history_df": history_df,
        "candidate_df": candidate_df,
        "model": best_model,
        "best_params": best_params,
        "best_threshold": best_threshold,
        "valid_metrics": best_metrics,
        "coef_df": coef_df
    }

## StepWise

In [10]:
def stepwise_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    scaler_type="standard",
    threshold_search=True,
    threshold_grid=None,
    param_grid=None,
    min_delta=0.0,
    max_steps=None,
    max_features=None,
    random_state=1,
    verbose=True
):
    """
    Stepwise Selection + LogisticRegression(L2) 하이퍼파라미터 서칭.
    매 step에서 추가를 먼저 하고, 이후 제거 가능성을 확인.
    train으로 학습하고 valid로 검증.
    선택 기준은 valid G-Mean.
    """

    if param_grid is None:
        param_grid = make_logistic_l2_param_grid(random_state=random_state)

    remaining_features = list(X_train.columns)
    selected_features = []

    current_score = 0.0
    scores = []
    history = []
    candidate_records = []

    best_model = None
    best_params = None
    best_threshold = 0.5
    best_metrics = None

    print("=" * 80)
    print("Stepwise Selection + Logistic Hyperparameter Search")
    print("=" * 80)

    step = 0

    while len(remaining_features) > 0:
        step += 1

        if max_steps is not None and step > max_steps:
            print("\nmax_steps 도달. 종료.")
            break

        if max_features is not None and len(selected_features) >= max_features:
            print("\nmax_features 도달. 종료.")
            break

        if verbose:
            print(f"\n[Step {step}]")
            print(f"현재 선택 변수: {len(selected_features)}개")
            print(f"남은 변수: {len(remaining_features)}개")
            print(f"현재 G-Mean: {current_score:.4f}")

        improved = False

        # ==================================================
        # 1. Forward step
        # ==================================================
        add_feature = None
        add_eval = None
        add_score = current_score

        for feature in remaining_features:
            test_features = selected_features + [feature]

            eval_result = evaluate_feature_subset(
                X_train,
                y_train,
                X_valid,
                y_valid,
                features=test_features,
                param_grid=param_grid,
                scaler_type=scaler_type,
                threshold_search=threshold_search,
                threshold_grid=threshold_grid,
                random_state=random_state
            )

            metrics = eval_result["metrics"]
            params = eval_result["params"]

            candidate_records.append({
                "step": step,
                "action": "try_add",
                "candidate_feature": feature,
                "features": list(test_features),
                "n_features": len(test_features),
                "valid_gmean": metrics["gmean"],
                "valid_accuracy": metrics["accuracy"],
                "valid_precision": metrics["precision"],
                "valid_recall": metrics["recall"],
                "valid_specificity": metrics["specificity"],
                "valid_f1": metrics["f1"],
                "valid_auc": metrics["auc"],
                "threshold": eval_result["threshold"],
                "C": params["C"],
                "solver": params["solver"],
                "class_weight": params["class_weight"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "tp": metrics["tp"]
            })

            if metrics["gmean"] > add_score + min_delta:
                add_feature = feature
                add_eval = eval_result
                add_score = metrics["gmean"]

        if add_feature is not None:
            selected_features.append(add_feature)
            remaining_features.remove(add_feature)

            best_model = add_eval["model"]
            best_params = add_eval["params"]
            best_threshold = add_eval["threshold"]
            best_metrics = add_eval["metrics"]
            current_score = best_metrics["gmean"]

            scores.append(current_score)
            improved = True

            history.append({
                "step": step,
                "action": "add",
                "feature": add_feature,
                "features": list(selected_features),
                "n_features": len(selected_features),
                "valid_gmean": best_metrics["gmean"],
                "valid_accuracy": best_metrics["accuracy"],
                "valid_precision": best_metrics["precision"],
                "valid_recall": best_metrics["recall"],
                "valid_specificity": best_metrics["specificity"],
                "valid_f1": best_metrics["f1"],
                "valid_auc": best_metrics["auc"],
                "threshold": best_threshold,
                "C": best_params["C"],
                "solver": best_params["solver"],
                "class_weight": best_params["class_weight"],
                "tn": best_metrics["tn"],
                "fp": best_metrics["fp"],
                "fn": best_metrics["fn"],
                "tp": best_metrics["tp"]
            })

            print(
                f"  + 추가: {add_feature} | "
                f"G-Mean: {best_metrics['gmean']:.4f} | "
                f"Recall: {best_metrics['recall']:.4f} | "
                f"Specificity: {best_metrics['specificity']:.4f} | "
                f"C: {best_params['C']} | solver: {best_params['solver']} | "
                f"class_weight: {best_params['class_weight']} | "
                f"threshold: {best_threshold:.2f}"
            )
        else:
            print("  + 추가해도 G-Mean 개선 없음")

        # ==================================================
        # 2. Backward step
        # ==================================================
        if len(selected_features) > 1:
            remove_feature = None
            remove_eval = None
            remove_score = current_score

            for feature in selected_features:
                test_features = [f for f in selected_features if f != feature]

                eval_result = evaluate_feature_subset(
                    X_train,
                    y_train,
                    X_valid,
                    y_valid,
                    features=test_features,
                    param_grid=param_grid,
                    scaler_type=scaler_type,
                    threshold_search=threshold_search,
                    threshold_grid=threshold_grid,
                    random_state=random_state
                )

                metrics = eval_result["metrics"]
                params = eval_result["params"]

                candidate_records.append({
                    "step": step,
                    "action": "try_remove",
                    "candidate_feature": feature,
                    "features": list(test_features),
                    "n_features": len(test_features),
                    "valid_gmean": metrics["gmean"],
                    "valid_accuracy": metrics["accuracy"],
                    "valid_precision": metrics["precision"],
                    "valid_recall": metrics["recall"],
                    "valid_specificity": metrics["specificity"],
                    "valid_f1": metrics["f1"],
                    "valid_auc": metrics["auc"],
                    "threshold": eval_result["threshold"],
                    "C": params["C"],
                    "solver": params["solver"],
                    "class_weight": params["class_weight"],
                    "tn": metrics["tn"],
                    "fp": metrics["fp"],
                    "fn": metrics["fn"],
                    "tp": metrics["tp"]
                })

                if metrics["gmean"] > remove_score + min_delta:
                    remove_feature = feature
                    remove_eval = eval_result
                    remove_score = metrics["gmean"]

            if remove_feature is not None:
                selected_features.remove(remove_feature)
                remaining_features.append(remove_feature)

                best_model = remove_eval["model"]
                best_params = remove_eval["params"]
                best_threshold = remove_eval["threshold"]
                best_metrics = remove_eval["metrics"]
                current_score = best_metrics["gmean"]

                scores.append(current_score)
                improved = True

                history.append({
                    "step": step,
                    "action": "remove",
                    "feature": remove_feature,
                    "features": list(selected_features),
                    "n_features": len(selected_features),
                    "valid_gmean": best_metrics["gmean"],
                    "valid_accuracy": best_metrics["accuracy"],
                    "valid_precision": best_metrics["precision"],
                    "valid_recall": best_metrics["recall"],
                    "valid_specificity": best_metrics["specificity"],
                    "valid_f1": best_metrics["f1"],
                    "valid_auc": best_metrics["auc"],
                    "threshold": best_threshold,
                    "C": best_params["C"],
                    "solver": best_params["solver"],
                    "class_weight": best_params["class_weight"],
                    "tn": best_metrics["tn"],
                    "fp": best_metrics["fp"],
                    "fn": best_metrics["fn"],
                    "tp": best_metrics["tp"]
                })

                print(
                    f"  - 제거: {remove_feature} | "
                    f"G-Mean: {best_metrics['gmean']:.4f} | "
                    f"Recall: {best_metrics['recall']:.4f} | "
                    f"Specificity: {best_metrics['specificity']:.4f} | "
                    f"C: {best_params['C']} | solver: {best_params['solver']} | "
                    f"class_weight: {best_params['class_weight']} | "
                    f"threshold: {best_threshold:.2f}"
                )
            else:
                print("  - 제거해도 G-Mean 개선 없음")

        if not improved:
            print("\n추가/제거 모두 개선 없음. 종료.")
            break

    history_df = pd.DataFrame(history)
    candidate_df = pd.DataFrame(candidate_records)

    if len(candidate_df) > 0:
        candidate_df = candidate_df.sort_values(
            by=["valid_gmean", "valid_recall", "valid_accuracy"],
            ascending=False
        ).reset_index(drop=True)

    coef_df = get_coef_df(best_model, selected_features) if best_model is not None else pd.DataFrame()

    return {
        "method": "stepwise",
        "selected_features": selected_features,
        "scores": scores,
        "history_df": history_df,
        "candidate_df": candidate_df,
        "model": best_model,
        "best_params": best_params,
        "best_threshold": best_threshold,
        "valid_metrics": best_metrics,
        "coef_df": coef_df
    }

## 출력

전역

In [ ]:
global_result = global_exhaustive_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    min_features=2,
    max_features=7,
    scaler_type="standard",
    threshold_search=True,
    random_state=1,
    verbose=True
)

print_selection_summary(global_result, title="전역선택법 결과")

print(f"Best AIC: {global_result['best_aic']:.2f}")
print(f"Best BIC: {global_result['best_bic']:.2f}")

display(global_result["result_df"].head(20))
display(global_result["coef_df"])

전역선택법 Global Exhaustive Selection + Logistic Hyperparameter Search
전체 변수 개수: 28
평가 변수 개수 범위: 2 ~ 7
총 subset 조합 수: 1,683,189
Logistic parameter 후보 수: 576


2 features:   0%|          | 0/378 [00:00<?, ?it/s]

RuntimeError: 모든 LogisticRegression 후보가 실패함.

전진

In [16]:


forward_result = forward_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    scaler_type="standard",
    threshold_search=True,
    min_delta=0.0,
    max_features=None,
    random_state=1,
    verbose=True
)

print_selection_summary(forward_result, title="전진선택법 결과")

display(forward_result["history_df"])
display(forward_result["candidate_df"].head(20))
display(forward_result["coef_df"])

전진선택법 Forward Selection + Logistic Hyperparameter Search

[Step 1] 현재 선택 변수 0개 / 남은 변수 28개
현재 Best Valid G-Mean: 0.0000


KeyboardInterrupt: 

후진

In [ ]:
backward_result = backward_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    scaler_type="standard",
    threshold_search=True,
    min_delta=0.0,
    min_features=1,
    random_state=1,
    verbose=True
)

print_selection_summary(backward_result, title="후진선택법 결과")

display(backward_result["history_df"])
display(backward_result["candidate_df"].head(20))
display(backward_result["coef_df"])

Valid G-Mean: 0.671121323325298
Selected Features: ['NASDAQ_return(%)', 'VKOSPI_Close', 'KOSPI 200_DMI14', 'USD/KRW_change(%)', 'Gold Spot_return(%)', 'KODEX 200_Premium', 'KOSPI 200_PPO_Hist', 'Signal3_Sell', 'Signal2_Buy', 'VKOSPI_Change(%)', 'KOSPI 200 Volume']


StepWise

In [ ]:
stepwise_result = stepwise_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    scaler_type="standard",
    threshold_search=True,
    min_delta=0.0,
    max_steps=None,
    max_features=None,
    random_state=1,
    verbose=True
)

print_selection_summary(stepwise_result, title="Stepwise 선택법 결과")

display(stepwise_result["history_df"])
display(stepwise_result["candidate_df"].head(20))
display(stepwise_result["coef_df"])

NameError: name 'stepwise_selection_lr' is not defined

In [ ]:
final_result = 

final_model = final_result["model"]
final_features = final_result["selected_features"]
final_threshold = final_result["best_threshold"]

y_test_prob = final_model.predict_proba(X_test[final_features])[:, 1]
y_test_pred = (y_test_prob >= final_threshold).astype(int)

test_metrics = classification_metrics(y_test, y_test_pred, y_test_prob)

test_metrics